In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

In [6]:
## -- Options --
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

## -- Accessories List --
Accessories = []
seen_links = set()  # to avoid duplicates

In [7]:
START_URL = "https://www.amazon.eg/s?k=mobile+accessories&i=electronics&s=exact-aware-popularity-rank"  # already sorted by Best Sellers

driver = webdriver.Chrome(options=options)
driver.get(START_URL)

## -- Accept Cookies --
try:
    WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.ID, "sp-cc-accept"))
    ).click()
except TimeoutException:
    pass

## -- Click "See all results" only if it exists (category pages only) --
try:
    WebDriverWait(driver, 3).until(
        EC.element_to_be_clickable((By.ID, "apb-desktop-browse-search-see-all"))
    ).click()
except TimeoutException:
    pass

In [8]:
## -- Scraping --
page = 1

while True:
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, 'div[data-component-type="s-search-result"]')
            )
        )
    except TimeoutException:
        print("No results found on this page (maybe a CAPTCHA?). Stopping.")
        break

    cards = driver.find_elements(By.CSS_SELECTOR, 'div[data-component-type="s-search-result"]')
    page_products = 0

    for card in cards:
        try:
            Name = card.find_element(By.CSS_SELECTOR, 'h2').get_attribute('aria-label')
            Link = card.find_element(By.CSS_SELECTOR, 'a.a-link-normal[href*="/dp/"]').get_attribute('href')
            Image = card.find_element(By.CSS_SELECTOR, 'img.s-image').get_attribute('src')
        except NoSuchElementException:
            continue

        clean_link = Link.split("?")[0]
        if clean_link in seen_links:
            continue
        seen_links.add(clean_link)

        try:
            Rating = card.find_element(By.CSS_SELECTOR, '[data-cy="reviews-block"] span.a-size-small.a-color-base').text
        except NoSuchElementException:
            Rating = "Not Available"

        try:
            Reviews = card.find_element(By.CSS_SELECTOR, '[data-cy="reviews-block"] a[aria-label$="ratings"]').get_attribute('aria-label')
        except NoSuchElementException:
            Reviews = "Not Available"

        try:
            Price = card.find_element(By.CLASS_NAME, 'a-price-whole').text
        except NoSuchElementException:
            Price = "Product Not Available"

        Accessories.append({
            'Name': Name,
            'Link': clean_link,
            'Image': Image,
            'Rating': Rating,
            'Reviews': Reviews,
            'Price': Price
        })
        page_products += 1

    print(
        f"Page {page} finished successfully. "
        f"Products collected on this page: {page_products}. "
        f"Total products collected: {len(Accessories)}"
    )

    try:
        next_page = driver.find_element(By.CSS_SELECTOR, 'a.s-pagination-next')

        if 's-pagination-disabled' in next_page.get_attribute('class'):
            print("Scraping finished. No more pages available.")
            break

        next_page.click()
        page += 1
        time.sleep(5)

    except NoSuchElementException:
        print("Next page button not found. Check the pagination selector.")
        break

driver.quit()

Page 1 finished successfully. Products collected on this page: 24. Total products collected: 24
Page 2 finished successfully. Products collected on this page: 24. Total products collected: 48
Page 3 finished successfully. Products collected on this page: 24. Total products collected: 72
Page 4 finished successfully. Products collected on this page: 24. Total products collected: 96
Page 5 finished successfully. Products collected on this page: 24. Total products collected: 120
Page 6 finished successfully. Products collected on this page: 24. Total products collected: 144
Page 7 finished successfully. Products collected on this page: 24. Total products collected: 168
Page 8 finished successfully. Products collected on this page: 24. Total products collected: 192
Page 9 finished successfully. Products collected on this page: 24. Total products collected: 216
Page 10 finished successfully. Products collected on this page: 24. Total products collected: 240
Page 11 finished successfully. Pr

In [9]:
df_Accessories = pd.DataFrame(Accessories)
df_Accessories.head()

,Name,Link,Image,Rating,Reviews,Price
0,LvShuLiangPin 1 Pack Silicone Suction Cup Phon...,https://www.amazon.eg/-/en/LvShuLiangPin-Silic...,https://m.media-amazon.com/images/I/614-6dpQVV...,3.0,962 ratings,32
1,"StickyGrippy Suction Phone Case Mount, Sillico...",https://www.amazon.eg/-/en/StickyGrippy-Sillic...,https://m.media-amazon.com/images/I/71gaz4iZB1...,3.1,"3,932 ratings",34
2,Joyroom JR-W020 20W PD 10000mAh Magnetic Wirel...,https://www.amazon.eg/-/en/Joyroom-JR-W020-100...,https://m.media-amazon.com/images/I/41OzmE-1fz...,3.7,237 ratings,899
3,LENOK PH33 360° Rotating Metal Phone Stand for...,https://www.amazon.eg/-/en/LENOK-PH33-Rotating...,https://m.media-amazon.com/images/I/51HMKriy1v...,4.8,385 ratings,199
4,"ONCHEER 360°Rotating Phone Holder for Desk, Po...",https://www.amazon.eg/-/en/ONCHEER-360%C2%B0Ro...,https://m.media-amazon.com/images/I/610yvHK7Z2...,3.5,"1,237 ratings",68


In [10]:
df_Accessories.to_csv("amazon_eg_mobile_accessories.csv", index=False, encoding="utf-8-sig")

In [1]:
import pandas as pd
df = pd.read_csv("amazon_eg_mobile_accessories.csv", encoding="utf-8-sig")
df.to_parquet("amazon_eg_mobile_accessories.parquet", index=False)